In [1]:
import os 
from itables import show
import pandas as pd
import plotly.express as px
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)
%matplotlib inline

from utils import read_triples_from_file

from utils import plot_box, get_table, prepare_data_for_plot

In [2]:
path = f"./results/exception_discovery_grid/"

In [3]:
exceptions_df = pd.read_csv(path+"exceptions_summary_ALL.csv")
#exceceptions_details_df = pd.read_csv(path+"exceptions_details_ALL.csv")

In [4]:
show(exceptions_df)

In [5]:
def parse_param(value: str):
    number = float(value.replace("_", "."))
    return int(number) if number.is_integer() else number


exceptions_df["mincov"] = (
    exceptions_df["model"]
    .str.extract(r"_mincov_([0-9_]+)_threshold")[0]
    .apply(parse_param)
)

exceptions_df["threshold"] = (
    exceptions_df["model"]
    .str.extract(r"_threshold_([0-9_]+)$")[0]
    .apply(parse_param)
)

In [6]:
exceptions_df

,dataset,model,thread_time,process_time,raw_time,liczba_regul,liczba_wyjatkow,srednia_dlugosc_reguly,suma_warunkow,avg_precision,avg_coverage,mincov,threshold
0,anneal.arff,algorithm_mincov_5_threshold_0_6,4.227058,4.227064,4.257991,27,4,1.333333,36,0.92,0.29,5.0,0.6
1,anneal.arff,algorithm_mincov_5_threshold_0_7,3.756536,3.756515,3.756554,28,2,1.500000,42,0.92,0.30,5.0,0.7
2,anneal.arff,algorithm_mincov_5_threshold_0_8,3.752654,3.752642,3.785624,28,2,1.500000,42,0.92,0.30,5.0,0.8
3,anneal.arff,algorithm_mincov_5_threshold_0_9,3.445390,3.445389,3.477280,30,2,1.933333,58,0.94,0.30,5.0,0.9
4,anneal.arff,algorithm_mincov_0_1_threshold_0_6,2.553198,2.553193,2.553371,19,4,1.894737,36,0.85,0.37,0.1,0.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
387,zoo.arff,algorithm_mincov_5_threshold_0_9,0.059985,0.059985,0.059984,11,0,2.545455,28,1.00,0.72,5.0,0.9
388,zoo.arff,algorithm_mincov_0_1_threshold_0_6,0.254597,0.254596,0.254606,14,0,2.571429,36,0.99,0.77,0.1,0.6
389,zoo.arff,algorithm_mincov_0_1_threshold_0_7,0.220660,0.220660,0.220659,14,0,2.571429,36,0.99,0.77,0.1,0.7
390,zoo.arff,algorithm_mincov_0_1_threshold_0_8,0.186362,0.186362,0.186361,14,0,2.571429,36,0.99,0.77,0.1,0.8


In [7]:
tabela = (
    exceptions_df
    .groupby(["mincov", "threshold"], as_index=False)
    .agg(
        liczba_regul=("liczba_regul", "sum"),
        liczba_wyjatkow=("liczba_wyjatkow", "sum")
    )
)


In [8]:
tabela

,mincov,threshold,liczba_regul,liczba_wyjatkow
0,0.1,0.6,940,13
1,0.1,0.7,944,7
2,0.1,0.8,968,5
3,0.1,0.9,991,3
4,5.0,0.6,1486,33
5,5.0,0.7,1549,15
6,5.0,0.8,1723,6
7,5.0,0.9,1909,5


In [9]:
pivot = exceptions_df.pivot_table(
    index="mincov",
    columns="threshold",
    values=["liczba_wyjatkow"],
    aggfunc="sum"
)

pivot

liczba_wyjatkow            
threshold             0.6 0.7 0.8 0.9
mincov                               
0.1                    13   7   5   3
5.0                    33  15   6   5

In [10]:
pivot_dataset = exceptions_df.pivot_table(
    index=["dataset", "mincov"],
    columns="threshold",
    values=["liczba_regul", "liczba_wyjatkow"],
    aggfunc="sum"
)

pivot_dataset

liczba_regul                liczba_wyjatkow            
threshold                      0.6  0.7  0.8  0.9             0.6 0.7 0.8 0.9
dataset        mincov                                                        
anneal.arff    0.1              19   19   19   20               4   2   2   2
               5.0              27   28   28   30               4   2   2   2
audiology.arff 0.1              56   56   56   56               0   0   0   0
               5.0              43   43   43   43               0   0   0   0
auto-mpg.arff  0.1              20   21   21   21               0   0   0   0
...                            ...  ...  ...  ...             ...  ..  ..  ..
wine.arff      5.0               9    9    9    9               0   0   0   0
yeast.arff     0.1              69   71   71   71               0   0   0   0
               5.0             175  188  202  206               3   3   0   0
zoo.arff       0.1              14   14   14   14               0   0   0   0
               5.0              11   11   11   11               0   0   0   0

[98 rows x 8 columns]

In [11]:

podsumowanie_wyjatkow = (
    exceptions_df[exceptions_df["liczba_wyjatkow"] > 0]
    .groupby(["mincov", "threshold"])["dataset"]
    .nunique()
    .reset_index(name="liczba_zbiorow_z_wyjatkami")
)

podsumowanie_wyjatkow

,mincov,threshold,liczba_zbiorow_z_wyjatkami
0,0.1,0.6,6
1,0.1,0.7,4
2,0.1,0.8,3
3,0.1,0.9,2
4,5.0,0.6,13
5,5.0,0.7,8
6,5.0,0.8,4
7,5.0,0.9,2


In [12]:
tmp = exceptions_df[exceptions_df["liczba_wyjatkow"] > 0]

tabela_wyjatkow = tmp.pivot_table(
    index="mincov",
    columns="threshold",
    values="raw_time",
    aggfunc="mean",
    fill_value=0
).astype(int)

tabela_wyjatkow

threshold,0.6,0.7,0.8,0.9
mincov,,,,
0.1,43,25,22,2
5.0,304,142,32,4


In [13]:
tabela_czasow = tmp.pivot_table(
    index="mincov",
    columns="threshold",
    values="dataset",
    aggfunc="nunique",
    fill_value=0
).astype(int)


In [14]:
podsumowanie_datasetow = (
    exceptions_df[exceptions_df["liczba_wyjatkow"] > 0]
    .groupby("dataset", as_index=False)
    .agg(
        liczba_wyjatkow=("liczba_wyjatkow", "sum"),
        liczba_konfiguracji_z_wyjatkami=("model", "nunique")
    )
    .sort_values("liczba_wyjatkow", ascending=False)
)

podsumowanie_datasetow

,dataset,liczba_wyjatkow,liczba_konfiguracji_z_wyjatkami
0,anneal.arff,20,8
9,mushroom.arff,18,8
12,sonar.arff,8,6
3,credit-g.arff,7,2
13,vehicle.arff,7,2
14,yeast.arff,6,2
4,cylinder-bands.arff,5,3
8,kdd-synthetic-control.arff,4,2
5,diabetes.arff,3,2
10,segment.arff,3,1


In [15]:
exceceptions_details_df = pd.read_csv(path+"exceptions_details_ALL.csv")

In [16]:
exceceptions_details_df

,dataset,model,CR_number,ER_number,CR,RR,ER,GACE,RI,MY_MEASURE
0,anneal.arff,algorithm_mincov_5_threshold_0_6,0,0,IF surface-quality = {E} THEN class = 3 (p=297...,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} AND carbon >= 3.50 TH...,0.027044,4.612511,0.505567
1,anneal.arff,algorithm_mincov_5_threshold_0_6,1,0,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} THEN class = 3 (p=297...,IF carbon >= 3.50 AND surface-quality = {E} TH...,0.014868,5.311476,0.554094
2,anneal.arff,algorithm_mincov_5_threshold_0_6,2,0,"IF width >= 1,410.05 THEN class = 2 (p=14, n=2...","IF strength >= 450.00 THEN class = 2 (p=19, n=...","IF width >= 1,410.05 AND strength >= 450.00 TH...",0.004197,3.663894,0.445707
3,anneal.arff,algorithm_mincov_5_threshold_0_6,3,0,"IF strength >= 450.00 THEN class = 2 (p=19, n=...","IF width >= 1,410.05 THEN class = 2 (p=14, n=2...","IF strength >= 450.00 AND width >= 1,410.05 TH...",0.004060,1.642777,0.368687
4,anneal.arff,algorithm_mincov_5_threshold_0_7,0,0,IF surface-quality = {E} THEN class = 3 (p=297...,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} AND carbon >= 3.50 TH...,0.027044,4.612511,0.505567
...,...,...,...,...,...,...,...,...,...,...
82,yeast.arff,algorithm_mincov_5_threshold_0_6,1,0,IF alm >= 0.53 AND mit <= 0.13 AND mcg >= 0.49...,"IF nuc >= 0.37 THEN class = NUC (p=128, n=83, ...",IF alm >= 0.53 AND mit <= 0.13 AND mcg >= 0.49...,0.002228,2.422353,0.322990
83,yeast.arff,algorithm_mincov_5_threshold_0_6,2,0,IF alm >= 0.53 AND mit <= 0.30 AND vac <= 0.52...,"IF nuc >= 0.48 THEN class = NUC (p=60, n=23, P...",IF alm >= 0.53 AND mit <= 0.30 AND vac <= 0.52...,0.003659,1.960147,0.299425
84,yeast.arff,algorithm_mincov_5_threshold_0_7,0,0,IF alm >= 0.53 AND mit <= 0.13 AND mcg >= 0.49...,"IF nuc >= 0.48 THEN class = NUC (p=60, n=23, P...",IF alm >= 0.53 AND mit <= 0.13 AND mcg >= 0.49...,0.002703,2.929748,0.385490
85,yeast.arff,algorithm_mincov_5_threshold_0_7,1,0,IF alm >= 0.47 AND mit <= 0.25 AND nuc <= 0.31...,"IF alm >= 0.69 THEN class = CYT (p=12, n=3, P=...",IF alm >= 0.47 AND mit <= 0.25 AND nuc <= 0.31...,0.003599,2.647119,0.346292


In [17]:
exceceptions_details_df_article = exceceptions_details_df[["dataset", "CR", "RR", "ER"]]

In [18]:
exceceptions_details_df_article[(exceceptions_details_df_article["dataset"] == "anneal.arff") | (exceceptions_details_df_article["dataset"] == "mushroom.arff")]

,dataset,CR,RR,ER
0,anneal.arff,IF surface-quality = {E} THEN class = 3 (p=297...,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} AND carbon >= 3.50 TH...
1,anneal.arff,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} THEN class = 3 (p=297...,IF carbon >= 3.50 AND surface-quality = {E} TH...
2,anneal.arff,"IF width >= 1,410.05 THEN class = 2 (p=14, n=2...","IF strength >= 450.00 THEN class = 2 (p=19, n=...","IF width >= 1,410.05 AND strength >= 450.00 TH..."
3,anneal.arff,"IF strength >= 450.00 THEN class = 2 (p=19, n=...","IF width >= 1,410.05 THEN class = 2 (p=14, n=2...","IF strength >= 450.00 AND width >= 1,410.05 TH..."
4,anneal.arff,IF surface-quality = {E} THEN class = 3 (p=297...,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} AND carbon >= 3.50 TH...
5,anneal.arff,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} THEN class = 3 (p=297...,IF carbon >= 3.50 AND surface-quality = {E} TH...
6,anneal.arff,IF surface-quality = {E} THEN class = 3 (p=297...,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} AND carbon >= 3.50 TH...
7,anneal.arff,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} THEN class = 3 (p=297...,IF carbon >= 3.50 AND surface-quality = {E} TH...
8,anneal.arff,IF surface-quality = {E} THEN class = 3 (p=297...,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} AND carbon >= 3.50 TH...
9,anneal.arff,"IF carbon >= 3.50 THEN class = 3 (p=74, n=2, P...",IF surface-quality = {E} THEN class = 3 (p=297...,IF carbon >= 3.50 AND surface-quality = {E} TH...


## Podsumowanie znalezionych wyjątków

### Wyjątki znaleziono dla zbiorów

In [19]:
exceptions_df = exceptions_df[exceptions_df["liczba_wyjatkow"] > 0][["dataset", "model", "liczba_regul","liczba_wyjatkow" ]]
show(exceptions_df)

### Szczegóły znalezionych wyjątków

In [20]:
show(exceceptions_details_df)

In [21]:
datasets = exceptions_df["dataset"].values

for dataset in datasets:
    display(f"****************************************************************************")
    display(f"Dataset: {dataset}")
    read_triples_from_file(path+f"{dataset[:-5]}/algorithm/rules.txt", draw = True, problem_type="regression")
    display(f"****************************************************************************")
    print("\n\n")

'****************************************************************************'

'Dataset: anneal.arff'

FileNotFoundError: [Errno 2] No such file or directory: './results/exception_discovery_grid/anneal/algorithm/rules.txt'